# NMF でトピックを抽出する

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/12_topics_keywords/nmf_topics.ipynb)

NLP クックブック 12-02 ／ トピック・キーワード・要約

LDA より速く、短い文書でも安定しやすい方法でトピックを抽出したい。TF-IDF 行列を非負値行列因子分解する。

In [1]:
from sklearn.datasets import fetch_20newsgroups
from sklearn.decomposition import NMF
from sklearn.feature_extraction.text import TfidfVectorizer

data = fetch_20newsgroups(subset="train", remove=("headers", "footers", "quotes"))
docs = [d for d in data.data if len(d) > 200]
vec = TfidfVectorizer(stop_words="english", min_df=5, max_df=0.5, max_features=20000, token_pattern=r"(?u)\b[a-zA-Z]{3,}\b")
X = vec.fit_transform(docs)

nmf = NMF(n_components=12, init="nndsvd", random_state=0, max_iter=400)
W = nmf.fit_transform(X)      # 文書 × トピックの重み
H = nmf.components_           # トピック × 単語の重み
vocab = vec.get_feature_names_out()
for k, row in enumerate(H):
    print(f"{k:2} {' '.join(vocab[i] for i in row.argsort()[::-1][:9])}")

 0 don people think just like know right time say
 1 windows dos file files program use version disk running
 2 geb cadre dsl pitt chastity shameful intellect skepticism surrender
 3 god jesus bible christ faith christian believe christians church
 4 key chip encryption clipper keys escrow government algorithm law
 5 drive scsi drives disk hard ide controller floppy mac
 6 team game games year players hockey season play league
 7 mail thanks edu space information email com send info
 8 israel armenian armenians jews israeli turkish arab war people
 9 card video monitor bus cards drivers vga memory ram
10 car bike new cars price engine miles good dealer
11 window motif application server display widget manager use problem


In [2]:
# 各トピックを最もよく代表する文書
for k in [0, 1, 2]:
    i = W[:, k].argmax()
    print(k, "|", " ".join(docs[i].split())[:110])

0 | Accounts of Anti-Armenian Human Rights Violations in Azerbaijan #007 Prelude to Current Events in Nagorno-Kara
1 | This is the official Request for Discussion (RFD) for the creation of two new newsgroups for Microsoft Windows
2 | So just what was it you wanted to say? -- --------------------------------------------------------------------


- TF-IDF 行列 X を、非負の 2 つの行列の積 W × H で近似する。非負なので「足し算だけ」で文書が表現され、解釈しやすい。
- 確率モデルではないので、LDA のような確率的な解釈はできないが、決定的 (`init="nndsvd"`) で再現性が高い。
- ツイートやタイトルのような短文には、LDA より NMF のほうが良い結果になることが多い。